In [53]:
import open3d as o3d
import cv2
import numpy as np
import glob
from scipy.spatial import KDTree
import os

folder = "/home/ashutosh/Documents/ICP/yoyo"
fx = 525.0  # focal length x 
fy = 525.0  # focal length y 
cx = 319.5  # optical center x 
cy = 239.5  # optical center y
K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=np.float32)


In [54]:

def ICP(source, target, iterations=20, rotation_matrix=np.eye(3), translation_vector=np.zeros(3)):

    tree = KDTree(target)
    for i in range(iterations):
        # Step 1: Transform source
        transformed_source = (rotation_matrix @ source.T).T + translation_vector

        # Step 2: Find correspondences
        distance, indices = tree.query(transformed_source)
        closest_points = target[indices]

        # Step 3: Estimating Transformation
        centroid_src = np.mean(transformed_source, axis=0)
        centroid_tgt = np.mean(closest_points, axis=0)

        src_centered = transformed_source - centroid_src
        tgt_centered = closest_points - centroid_tgt

        # Compute covariance and SVD
        H = src_centered.T @ tgt_centered
        U, _, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt.T @ U.T
        t = centroid_tgt - R @ centroid_src

        # Step 6: Update total transform
        rotation_matrix = R @ rotation_matrix
        translation_vector = R @ translation_vector + t
        # print(rotation_matrix)
    
    return rotation_matrix, translation_vector 

In [55]:
def depth_to_point_cloud(depth, K, image_bgr=None):
    H, W = depth.shape
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]

    # Create meshgrid of pixel coordinates
    u = np.arange(W)
    v = np.arange(H)
    u_grid, v_grid = np.meshgrid(u, v, indexing='xy')  # (H, W)

    # Unproject depth to 3D points
    z = depth
    x = (u_grid - cx) * z / fx
    y = (v_grid - cy) * z / fy

    points = np.stack([x, y, z], axis=-1).reshape(-1, 3)

    # Optional: get colors from image
    colors = None
    if image_bgr is not None:
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        colors = image_rgb.reshape(-1, 3).astype(np.float32) / 255.0

    return points, colors


In [ ]:
points, _ = depth_to_point_cloud(image, K)    
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

o3d.visualization.draw_geometries([pcd])

In [58]:
for i, file in enumerate(glob.glob(folder)):
    img1 = f"{folder}/{i}.png"
    img2 = f"{folder}/{i+1}.png"
    image1 = cv2.imread(img1, cv2.IMREAD_UNCHANGED)
    image2 = cv2.imread(img2, cv2.IMREAD_UNCHANGED)

    source_points, _ = depth_to_point_cloud(image1, K) 
    target_points, _ = depth_to_point_cloud(image2, K) 

    R_est, t_est = ICP(
    source=source_points,
    target=target_points,
    rotation_matrix=np.eye(3),
    translation_vector=np.zeros(3),
    iterations=25
    )

    

KeyboardInterrupt: 